In [1]:
import pandas as pd
import numpy as np
import os
datadir = 'files'

file = os.path.join(datadir, 'discentes.parquet')
discentes = pd.read_parquet(file)

# Tabela 1 Discente (pessoa)
# Garantir unicidade de linha por id_discente
discentes.drop_duplicates(subset=['id_discente'], inplace=True, ignore_index=True)

# Tabela 2 Matricula (componente matriculado)
file = os.path.join(datadir, 'matriculas.parquet')
matriculas = pd.read_parquet(file)

# Tabela 3 Componentes (componente matriculado)
file = os.path.join(datadir, 'componentes.parquet')
componentes = pd.read_parquet(file)

# Tabela 4 Cursos (curso)
file = os.path.join(datadir, 'cursos.parquet')
cursos = pd.read_parquet(file)

# Tabela 5 Docentes (curso)
file = os.path.join(datadir, 'docentes.parquet')
docentes = pd.read_parquet(file)
# TODO: não existe relacionamento com as outras tabelas por IDs, alternativa: criar um agrupamento em nivel de departamento e fazer a junção

# display(discentes.info())
# discentes.status_discente.value_counts()

# matriculas # id_discente	id_disciplina

# discentes # id_discente id_curso id_estrutura_curricular

# print(discentes.id_discente.count())
# print(discentes.id_discente.nunique())

# componentes
# print(componentes.id_disciplina.count())
# print(componentes.id_disciplina.nunique())

# cursos.id_curso.nunique()
print(len(cursos))
cursos[['id_estrutura_curricular', 'id_disciplina']].drop_duplicates().count()

matriculas

1310


,id_discente,id_disciplina,ano,periodo,situacao
0,9519063299906826,25772,2020,1,APROVADO
1,9519063299906826,25774,2020,1,REPROVADO
2,9519063299906826,25773,2020,1,REPROVADO
3,9519063299906826,25819,2020,1,REP. MEDIA E FALTA
4,9519063299906826,25815,2020,1,REP. MEDIA E FALTA
...,...,...,...,...,...
426748,8520061087277152,27240,2023,1,APROVADO
426749,8520061087277152,27242,2023,1,APROVADO
426750,8520061087277152,27239,2023,1,APROVADO
426751,8520061087277152,27241,2023,1,APROVADO


In [2]:
status_df = (
    matriculas.merge(discentes, how='left', on = 'id_discente')
    .merge(componentes, how='left', on='id_disciplina')
    .merge(cursos, how='left', on=['id_curso', 'id_estrutura_curricular', 'id_disciplina'], suffixes=['', '_curso'])
    )
status_df.columns

Index(['id_discente', 'id_disciplina', 'ano', 'periodo', 'situacao', 'sexo',
       'estado_civil', 'raca_declarada', 'discente_nivel', 'id_curso',
       'id_curriculo', 'id_estrutura_curricular', 'ano_ingresso',
       'periodo_ingresso', 'status_discente', 'forma_ingresso',
       'quantidade_membros_familia', 'ch_integralizada', 'ch_pendente',
       'media_geral', 'ano_nascimento', 'faixa_renda_familiar',
       'uf_titulo_eleitor_pb', 'uf_naturalidade_pb', 'pais_origem_br',
       'id_detalhe', 'nome', 'ch_aula', 'ch_laboratorio', 'ch_total',
       'cr_aula', 'cr_laboratorio', 'cr_estagio', 'ch_ead',
       'sigla_departamento', 'nivel_componente_curricular', 'sigla_academica',
       'nome_departamento', 'sigla_centro', 'nome_centro',
       'qtd_max_matriculas', 'codigo_componente_curricular',
       'nome_componete_curricular', 'descricao_tipo_componente_curricular',
       'excluir_avaliacao_institucional', 'ativo', 'curso_nome',
       'curso_unidade_nome', 'campus', 'curso

## Objetivo
Desenvolver um modelo preditivo para prever a evasão de curso técnico de um aluno na UFPB.




## Engenharia de features

In [4]:
aprovdisc = (
    status_df[['situacao', 'id_disciplina', 'id_discente']]
    .assign(aprov_disciplinas = lambda x: x.situacao.str.contains("^APROV").astype(int))
    .groupby(['id_discente'])
    .agg({"aprov_disciplinas": "mean"})
    .reset_index()
)

In [27]:
# target 1, de maneira simples usaremos a média do aluno, o ruim é que não captura tendência
# usando o primeiro quartil, ou seja, 25% dos alunos de tal curso tem a pior
status_df['percentil_media'] = (
    status_df.groupby(['curso_nome', 'ano', 'periodo'])['media_geral']
      .rank(pct=True)
)

status_df['target_percentil_media'] = (status_df['percentil_media'] <= 0.25).astype(int)

In [38]:
# analisando por tamanho qual a melhor composição para agregar pela média
print(status_df.groupby(['curso_nome']).size().describe())
print(status_df.groupby(['curso_nome', 'periodo']).size().describe())
print(status_df.groupby(['curso_nome', 'ano', 'periodo']).size().describe())

count       21.000000
mean      6879.047619
std       6084.001442
min         56.000000
25%       2180.000000
50%       4908.000000
75%      11333.000000
max      18542.000000
dtype: float64
count       39.000000
mean      3704.102564
std       3017.022528
min         56.000000
25%       1495.500000
50%       2580.000000
75%       5764.500000
max      11074.000000
dtype: float64
count     281.000000
mean      514.092527
std       389.350538
min         1.000000
25%       214.000000
50%       432.000000
75%       722.000000
max      2001.000000
dtype: float64


In [20]:
# from unidecode import unidecode
df = status_df.copy()
df = df.dropna(subset=['status_discente'])

# Definir o Y
df["evasao"] = (df["status_discente"] == "CANCELADO").astype(int)

# Criar variaveis
df["idade_ingresso"] = df["ano_ingresso"] - df["ano_nascimento"]
df["progresso_curso"] = df["ch_integralizada"] / (df["ch_integralizada"] + df["ch_pendente"])
df["carga_restante"] = df["ch_pendente"]
df["ingresso_recente"] = (df["ano_ingresso"] >= 2020).astype(int)


# X da dimensão discente
x1 = (df
.dropna(subset=['status_discente', 'idade_ingresso'])
.drop_duplicates(subset='id_discente', ignore_index=True)
.merge(aprovdisc)
.astype({"id_curso": int})
)


# Funções auxiliares
def gera_dummies(df, colname, map=None):
    if map is not None:
        df[colname] = df[colname].map(map)
    else:
        df[colname] = df[colname].astype(str).str.lower()
    return pd.get_dummies(df, columns=[colname], drop_first=True, dtype=int)
    
# Dummies de Sexo
x1 = gera_dummies(x1, 'sexo',  {"F": "feminino", "M": "masculino"})
x1 = gera_dummies(x1, 'estado_civil',  {"Solteiro(a)": "solteiro", "Casado(a)": "casado", "Outro": "outro"})
for col in ['raca_declarada', 'faixa_renda_familiar', 'id_curso']:
    x1 = gera_dummies(x1, col)

# # 'quantidade_membros_familia', 'progresso_curso', 'carga_restante': tem muito NA
colunas_base = ['evasao', 'ano_ingresso', 'periodo_ingresso', 
               'idade_ingresso',  'aprov_disciplinas', 'ingresso_recente']

# pegar automaticamente as dummies
colunas_dummies = [
    c for c in x1.columns
    if c.startswith("sexo_")
    or c.startswith("estado_civil_")
    or c.startswith("raca_declarada_")
    or c.startswith("faixa_renda_familiar_")
    or c.startswith("id_curso_")
]

data = x1[colunas_base+colunas_dummies]

# =========================================================
#  DIVISÃO TREINO / TESTE TEMPORAL
# =========================================================
CUTOFF = 2020

# Excluir os dois ultimos anos de ingressantes para evitar problema do tempo de curso
ANO_ATUAL = data.ano_ingresso.max()
data = data[data["ano_ingresso"] <= ANO_ATUAL - 2]

train = data[data["ano_ingresso"] <= CUTOFF].copy()
test  = data[data["ano_ingresso"] > CUTOFF].copy()

print("proporção treino:", len(train) / len(data))
print("proporção teste :", len(test) / len(data))

print("evasão treino:", train["evasao"].mean())
print("evasão teste :", test["evasao"].mean())

proporção treino: 0.5991448423303046
proporção teste : 0.40085515766969537
evasão treino: 0.5495093666369313
evasão teste : 0.4697777777777778


In [ ]:
# Bloco. X, y
X_train = train.drop(['evasao'], axis=1)
y_train = train['evasao']
X_test = test.drop(['evasao'], axis=1)
y_test = test['evasao']

In [28]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    f1_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Modelo
cv = StratifiedKFold(n_splits = 5, shuffle=True, random_state=1)

# (A) LOGIT
pipe_logit = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=1))
    ])

param_grid = {"model__C": [0.01, 0.1, 1, 10],
              "model__penalty": ["l1", "l2"]}
grid_logit = GridSearchCV(pipe_logit, param_grid=param_grid, cv=cv)
grid_logit.fit(X_train, y_train)


/opt/homebrew/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/homebrew/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/homebrew/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' 

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...om_state=1))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.01, 0.1, ...], 'model__penalty': ['l1', 'l2']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 :

In [31]:
print("Melhores parâmetros modelo Logit:", grid_logit.best_params_)
print("Melhor Scores de Performance:", grid_logit.best_score_)

Melhores parâmetros modelo Logit: {'model__C': 10, 'model__penalty': 'l2'}
Melhor Scores de Performance: 0.9107991933772024


In [32]:
# (B) Random Forest
pipe_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(random_state=1, class_weight="balanced"))
    ])

param_grid = {"model__n_estimators": [200, 400],
              "model__max_depth": [4, 8, None],
              "model__min_samples_leaf": [1, 5, 10]}
grid_rf = GridSearchCV(pipe_rf, param_grid=param_grid, cv=cv)
grid_rf.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...om_state=1))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [4, 8, ...], 'model__min_samples_leaf': [1, 5, ...], 'model__n_estimators': [200, 400]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fo

In [33]:
print("Melhores parâmetros modelo RF:", grid_rf.best_params_)
print("Melhor Scores de Performance:", grid_rf.best_score_)

Melhores parâmetros modelo RF: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 400}
Melhor Scores de Performance: 0.9158529682303828


In [42]:
# Bloco de seleção do melhor modelo entre as diferentes especificações
if np.round(grid_logit.best_score_,2) >= np.round(grid_rf.best_score_,2):
    melhor_modelo = grid_logit.best_estimator_
    nome_modelo = "Regressão Logística"
else:
    melhor_modelo = grid_rf.best_estimator_
    nome_modelo = "Floresta Aleatória"

print(nome_modelo)

# Definir o Cutoff da classificação: por padrão é usado 0.5 ou 50%
# Definir do threshold de forma dinâmica
prob_train = melhor_modelo.predict_proba(X_train)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_train, prob_train)
f1_scores = 2 * (precision * recall) / (precision + recall)
best_threshold = thresholds[np.argmax(f1_scores)]
print("Melhor Threshold:", best_threshold)

Floresta Aleatória
Melhor Threshold: 0.41022634148003057


In [48]:
# Avaliação no Teste
prob_test = melhor_modelo.predict_proba(X_test)[:, 1]
pred_test = (prob_test >= best_threshold).astype(int)
matriz_confusao = confusion_matrix(y_test, pred_test)
print(roc_auc_score(y_test, pred_test))
print(classification_report(y_test, pred_test))

0.851212647729859
              precision    recall  f1-score   support

           0       0.93      0.77      0.84      1193
           1       0.78      0.93      0.85      1057

    accuracy                           0.85      2250
   macro avg       0.86      0.85      0.85      2250
weighted avg       0.86      0.85      0.85      2250



In [51]:
# A importância dos atributos
if nome_modelo=="Regressão Logística":
    coefs = melhor_modelo.named_steps["model"].coef_[0]
    importancia = pd.DataFrame({"variavel": X_train.columns, 
    "coeficiente": coefs, "coef_abs": np.abs(coef)}).sort_values("coef_abs", ascending=False)
else:
    importancias = melhor_modelo.named_steps["model"].feature_importances_
    importancia = pd.DataFrame({"variavel": X_train.columns, 
    "importancia": importancias}).sort_values("importancia", ascending=False)

print(importancia)

                        variavel  importancia
3              aprov_disciplinas     0.673449
2                 idade_ingresso     0.076549
15      faixa_renda_familiar_nan     0.046318
0                   ano_ingresso     0.038300
14   faixa_renda_familiar_ate_1k     0.022198
5                 sexo_masculino     0.014489
1               periodo_ingresso     0.014044
9           raca_declarada_negra     0.012579
23              id_curso_1997022     0.009015
8   raca_declarada_nao_informado     0.008101
7          estado_civil_solteiro     0.008080
6             estado_civil_outro     0.007254
4               ingresso_recente     0.007179
25              id_curso_1997026     0.006919
19              id_curso_1958832     0.006847
18              id_curso_1958830     0.006559
36              id_curso_2663867     0.005248
35              id_curso_2558681     0.004891
20              id_curso_1996990     0.003963
11    faixa_renda_familiar_2k_4k     0.003852
51              id_curso_3926060  

In [55]:
# Salvar o modelo
import pickle
output = {
    "modelo": melhor_modelo, 
    "threshold": float(best_threshold),
    "features": X_train.columns.tolist(),
    "nome_modelo": nome_modelo,
    "importancia": importancia
}
with open("modelo_evasao.pkl", "wb") as file:
    pickle.dump(output, file)